# LLMGraphTransformer: open-schema extraction

Run cells in order from the repository root. Requires Ollama with the configured model.

This experiment extracts a graph, merges chunk duplicates, and exports JSON and an interactive HTML graph.

## 1. Configuration and PDF loading

Edit the model, PDF, and chunk settings here.

In [ ]:
from pathlib import Path
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from graph_io import export_graph, graph_document_records, visualize_graph

ROOT = Path.cwd()
PDF_PATH = ROOT / "abel-and-his-great-adventure.pdf"
OUTPUT_DIR = ROOT / "outputs"
MODEL = os.getenv("OLLAMA_MODEL", "gpt-oss:20b")
CHUNK_SIZE = 4000
CHUNK_OVERLAP = 200

if not PDF_PATH.is_file():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}. Start Jupyter from the repository root.")
pages = PyPDFLoader(str(PDF_PATH)).load()
if not pages:
    raise ValueError("The PDF contains no pages.")
print(f"Loaded {len(pages)} pages")

## 2. Extract and merge

Merging matches IDs ignoring case and surrounding whitespace, with exact node types. It does not resolve aliases or reconcile different types. The merged source is only the first chunk; per-entity provenance is not retained.

In [ ]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_ollama import ChatOllama
from merge_graph_documents import merge_graph_documents

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = splitter.split_documents(pages)
if not chunks:
    raise ValueError("No text chunks were produced. Scanned PDFs may require OCR.")
model = ChatOllama(model=MODEL, temperature=0)
transformer = LLMGraphTransformer(llm=model)

graph_documents = transformer.convert_to_graph_documents(chunks)
merged = merge_graph_documents(graph_documents)
print(f"Chunks: {len(chunks)}")
print(f"Nodes before merge: {sum(len(g.nodes) for g in graph_documents)}")
print(f"Merged: {len(merged.nodes)} nodes, {len(merged.relationships)} relationships")

## 3. Export and visualize

Exports use the merged graph. Open the generated HTML file in a browser.

In [ ]:
nodes, relationships = graph_document_records(merged)
export_graph(nodes, relationships, OUTPUT_DIR, "llm_transformer")
visualize_graph(nodes, relationships, OUTPUT_DIR / "llm_transformer_graph.html")